# Two beeps, and the moment they stop being one thing

Play a low beep and a high beep over and over, taking turns, and you hear two
separate things: a low stream and a high stream, running side by side. Play the
same two beeps *at the same time* and you hear one thing, a single sound with a
low part and a high part. That is true even when the two are more than an
octave apart, which is not what a textbook built on tonotopy predicts.

That is the finding in

> Elhilali M, Ma L, Micheyl C, Oxenham AJ, Shamma SA (2009).
> Temporal coherence in the perceptual organization and cortical representation
> of auditory scenes. *Neuron* 61:317-329.

The paper tests two extremes, together and taking turns. This notebook builds
both of them, plus everything in between: slide the high beep a little later
and a little later, and listen for where the one thing turns into two.

Everything here is the real experiment code. What you hear is what a subject
hears.

**Headphones. Runtime > Run all. Setup takes about 20 seconds.**

In [ ]:
# @title Setup: fetch the code and warm up (press play, about 20 s)
import os, subprocess, sys

REPO = "https://github.com/MeysamAmirsardari/Rate_RNN.git"
if not os.path.isdir("Rate_RNN"):
    # sparse clone: the repo is large, the experiment is 80 kB
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "--filter=blob:none", "--sparse", REPO], check=True)
    subprocess.run(["git", "-C", "Rate_RNN", "sparse-checkout", "set",
                    "--no-cone", "/audios/*.py", "/audios/streaming/*.py"],
                   check=True)
sys.path.insert(0, os.path.abspath("Rate_RNN"))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, HTML, display

from audios.streaming.config import Design
from audios.streaming import stimulus as S
from audios.streaming import verify as V
from audios.streaming.track import Track, simulate, geomean

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

# The real experiment plays into the left ear only, as the paper did. Here it
# goes to both, so it works on laptop speakers and on one earbud.
D = Design(ear="both")
D.validate()

def seq(df_st=15.0, gap_a_ms=50.0, lag_ms=0.0, dt_ms=0.0, sign=1,
        b_only=False, d=None, phase=(0.0, 0.0)):
    """One sequence. Returns the waveform and the two sets of onsets."""
    d = d or D
    return S.interval(d, df_st=df_st, gap_a_ms=gap_a_ms, lag_ms=lag_ms,
                      dt_ms=dt_ms, sign=sign, b_only=b_only, phase=phase)

def player(y, gain=1.0, fs=None):
    return Audio(np.asarray(y) * gain, rate=fs or D.fs,
                 normalize=False)._repr_html_()

def gain_for(*ys):
    return 0.85 / max(float(np.abs(np.asarray(y)).max()) for y in ys)

def players(items, fs=None):
    """[(label, waveform), ...] at one shared gain, so levels are comparable."""
    g = gain_for(*[y for _, y in items])
    return HTML("".join(
        f"<div style='margin:2px 0 12px'><b>{lab}</b><br>{player(y, g, fs)}</div>"
        for lab, y in items))

def piano(ivs, titles, d=None, mark_target=True, width=9.0):
    """The schedule as a picture: A on the bottom, B on top, target in red."""
    d = d or D
    fig, ax = plt.subplots(len(ivs), 1, figsize=(width, 0.95 * len(ivs)),
                           sharex=True, sharey=True, squeeze=False,
                           constrained_layout=True)
    for a, iv, t in zip(ax[:, 0], ivs, titles):
        if not iv["b_only"]:
            for x in iv["a"]:
                a.add_patch(plt.Rectangle((x, 0.10), d.tone, 0.24, color="0.35"))
        for i, x in enumerate(iv["b"]):
            last = mark_target and i == len(iv["b"]) - 1
            a.add_patch(plt.Rectangle((x, 0.62), d.tone, 0.24,
                                      color="#d62728" if last else "0.35"))
        a.set_ylim(0, 1); a.set_yticks([0.22, 0.74]); a.set_yticklabels(["A", "B"])
        a.set_ylabel(t, rotation=0, ha="right", va="center", fontsize=8)
        for s in ("top", "right"): a.spines[s].set_visible(False)
    ax[-1, 0].set_xlabel("time (ms).  the red one is the target")
    plt.show()

print("ready\n")
print(D.summary())

---
## 1. The two beeps

Two pure tones. A is always 1000 Hz. B sits above it, and how far above is one
of the things the paper varies: 6, 9 or 15 semitones. Fifteen semitones is more
than an octave, 1000 against 2378 Hz, and that is the condition that matters,
because at that distance a tonotopic account says they cannot possibly be one
object.

Each tone is 100 ms long including a 10 ms fade in and out, so 80 ms of steady
tone in the middle. The fade is what stops it clicking.

In [ ]:
d = D
for df in d.df_st:
    print(f"  {df:>2.0f} st   B = {d.f_b(df):7.1f} Hz   "
          f"{V.erb_rate(d.f_b(df)) - V.erb_rate(d.f_a):4.1f} critical bands above A")

n = int(round(d.tone * d.fs / 1000))
one_a = S._tone(d, d.f_a, n)
display(players([("A alone, 1000 Hz, one 100 ms tone", np.tile(np.r_[one_a, np.zeros(n // 2)], 6))]
              + [(f"B alone, {d.f_b(df):.0f} Hz ({df:.0f} st above A)",
                  np.tile(np.r_[S._tone(d, d.f_b(df), n), np.zeros(n // 2)], 6))
                 for df in d.df_st]))

fig, ax = plt.subplots(1, 2, figsize=(9, 2.4), constrained_layout=True)
t = np.arange(n) / d.fs * 1000
ax[0].plot(t, one_a, lw=0.4, color="0.4")
ax[0].plot(t, np.abs(__import__("scipy.signal", fromlist=["hilbert"]).hilbert(one_a)),
           lw=1.4, color="#d62728")
ax[0].set_title("one tone, and its 10 ms fades", fontsize=9, loc="left")
ax[0].set_xlabel("ms")
f = np.fft.rfftfreq(n, 1 / d.fs)
for df in (0.0,) + tuple(d.df_st):
    ff = d.f_a if df == 0 else d.f_b(df)
    p = np.abs(np.fft.rfft(S._tone(d, ff, n))) ** 2
    ax[1].plot(f, 10 * np.log10(p / p.max() + 1e-12), lw=1)
ax[1].set_xlim(0, 4000); ax[1].set_ylim(-80, 3)
ax[1].set_xlabel("Hz"); ax[1].set_ylabel("dB")
ax[1].set_title("the gate is clean: nothing splatters", fontsize=9, loc="left")
for a in ax:
    for s in ("top", "right"): a.spines[s].set_visible(False)
plt.show()

---
## 2. Taking turns, or together

Here is the whole point of the paper in two sounds. Same two tones, same
number of them, same loudness. The only difference is whether they land at the
same time.

Listen to the first one and you hear a low beeping and a high beeping, two
things going on at once, and you can attend to either. Listen to the second and
there is one thing, beeping, with a colour to it.

Fifteen semitones apart. More than an octave.

In [ ]:
alt = seq(df_st=15.0, gap_a_ms=50.0, lag_ms=D.alternation_ms(),
          d=D.replace(mode="sweep"))
syn = seq(df_st=15.0, gap_a_ms=50.0, lag_ms=0.0, d=D.replace(mode="sweep"))
display(players([("taking turns: two streams", alt["y"]),
                 ("together: one stream", syn["y"])]))
piano([alt, syn], ["taking turns", "together"], d=D.replace(mode="sweep"),
      mark_target=False)

---
## 3. How do you measure that without asking?

You could ask people how many streams they hear. People are bad at that, and
worse, they are bad at it in ways that depend on what you told them the
experiment was about.

The trick in this paper is much better, and it is worth understanding before
anything else here makes sense.

**Move the last high beep slightly, and ask which sequence it was in.**

Here is why that works. If you hear the low and high beeps as *one* thing, the
last high beep has a partner: the low beep it is supposed to be simultaneous
with. Comparing two things that are part of the same object is easy, and people
get down to two or three milliseconds.

If you have already split them into *two streams*, that comparison is not
available to you. Timing across streams is not something the auditory system
gives you. All you can do is listen to the high stream on its own and ask
whether its rhythm stumbled, and that is worth ten to twenty milliseconds.

So the threshold reports the percept. A small number means one stream. A big
number means two. Nobody is ever asked an opinion.

In [ ]:
d = D
big = S.trial(d, df_st=15.0, gap_a_ms=50.0, lag_ms=0.0, dt_ms=30.0,
              b_only=False, seed=3, target=1, sign=1)[0]
small = S.trial(d, df_st=15.0, gap_a_ms=50.0, lag_ms=0.0, dt_ms=3.0,
                b_only=False, seed=3, target=1, sign=1)[0]
display(HTML("<p>Synchronous sequences. In each pair the first one has the last "
             "high beep moved. Start with the obvious one.</p>"))
display(players([("30 ms late  (obvious)", big[0]["y"]),
                 ("...and the same thing with nothing moved", big[1]["y"]),
                 ("3 ms late  (this is roughly threshold)", small[0]["y"]),
                 ("...and the same thing with nothing moved", small[1]["y"])]))
piano([big[0], big[1]], ["30 ms moved", "nothing moved"])

And now the same 3 ms shift, but in a sequence where the two streams have
already come apart. Same shift. Same tones. It should be much harder, and if the
logic holds it should be about as hard as the version with no low beeps at all.

In [ ]:
hard = S.trial(D, df_st=15.0, gap_a_ms=30.0, lag_ms=0.0, dt_ms=3.0,
               b_only=False, seed=3, target=1, sign=1)[0]
bonly = S.trial(D, df_st=15.0, gap_a_ms=50.0, lag_ms=0.0, dt_ms=3.0,
                b_only=True, seed=3, target=1, sign=1)[0]
display(players([("3 ms, streams already apart (30 ms A gap)", hard[0]["y"]),
                 ("...nothing moved", hard[1]["y"]),
                 ("3 ms, no low beeps at all", bonly[0]["y"]),
                 ("...nothing moved", bonly[1]["y"])]))

---
## 4. The four published conditions

Five precursor tones at each frequency, then one target at each. The B stream
always keeps a 50 ms gap between its tones. The A stream gets a gap of 30, 50 or
70 ms, and that is how the paper makes the streams asynchronous: a different
gap means a different tempo, so the two trains drift out of step.

The two trains are pinned so that the **target** pair is exactly simultaneous in
the unshifted sequence. That is a nice detail. It means the drift lands on the
precursors and the judged pair is always in the same place relative to the end,
so the listener is never comparing an event 850 ms in against one 750 ms in.

Watch what that does to the picture. At 30 ms the low stream starts late and
catches up. At 70 ms it starts early and falls behind. At 50 ms nothing drifts
at all.

In [ ]:
rows = [seq(df_st=15.0, gap_a_ms=g) for g in D.gap_a_ms]
rows.append(seq(df_st=15.0, gap_a_ms=50.0, b_only=True))
names = [f"A gap {g:.0f} ms" for g in D.gap_a_ms] + ["B only"]
piano(rows, names, mark_target=True)
display(players(list(zip(
    [f"A gap {g:.0f} ms" + ("   (synchronous)" if g == 50 else "") for g in D.gap_a_ms]
    + ["B tones only, the control"], [r["y"] for r in rows]))))

The published thresholds, geometric means over nine trained listeners in a
booth. This is the target we are trying to land on.

| A gap | 6 st | 9 st | 15 st |
|---|---|---|---|
| **50 ms (synchronous)** | **2.6** | **2.7** | **3.2** |
| 30 ms | 11.5 | 14.0 | 18.0 |
| 70 ms | 13.5 | 14.5 | 21.0 |
| B only | 14.0 | 14.5 | 16.5 |

Two things to notice. Synchronous is four to six times better than everything
else, at every separation including the one wider than an octave. And the two
asynchronous conditions are no better than having no A tones at all, which is
the real claim: once the streams have split, the A tones are worth nothing.

---
## 5. Try it yourself

Two sequences. One of them has the last high beep moved, early or late, by the
amount shown after you reveal. Pick, then open the answer. Re-run the cell for
a new one; the condition and the shift are drawn at random each time.

Start with `DT_MS = 20` and work down. Around 3 ms in the synchronous condition
is where a trained listener sits.

In [ ]:
import random

DT_MS = 20.0          # <- change me. try 20, then 8, then 3, then 1.5
CONDITION = "random"  # "random", "sync", "async", "b_only"

cond = CONDITION
if cond == "random":
    cond = random.choice(["sync", "async", "b_only"])
gap = {"sync": 50.0, "async": random.choice([30.0, 70.0]), "b_only": 50.0}[cond]
df = random.choice(D.df_st)
ivs, target = S.trial(D, df_st=df, gap_a_ms=gap, lag_ms=0.0, dt_ms=DT_MS,
                      b_only=(cond == "b_only"), seed=random.randrange(10 ** 6))
g = gain_for(ivs[0]["y"], ivs[1]["y"])
sign = "late" if ivs[target - 1]["sign"] > 0 else "early"

display(HTML(
    "<div style='margin:2px 0 12px'><b>Sequence 1</b><br>" + player(ivs[0]["y"], g) + "</div>"
    "<div style='margin:2px 0 12px'><b>Sequence 2</b><br>" + player(ivs[1]["y"], g) + "</div>"
    "<details style='margin-top:10px;padding:12px 16px;background:#f4f4f6;"
    "border-radius:8px;max-width:600px'>"
    "<summary style='cursor:pointer;font-weight:600'>Reveal</summary>"
    f"<p style='margin:10px 0 4px'>The moved one was "
    f"<b style='font-size:16px'>sequence {target}</b>.</p>"
    f"<p style='margin:4px 0;color:#555'>Its last high beep was <b>{DT_MS:g} ms "
    f"{sign}</b>. Condition: <b>{cond}</b>, A gap {gap:.0f} ms, "
    f"B {df:.0f} st above A.</p>"
    "<p style='margin:10px 0 0;color:#888;font-size:12px'>Re-run for another."
    "</p></details>"))
piano([ivs[0], ivs[1]], ["Sequence 1", "Sequence 2"])

---
## 6. The staircase

Nobody measures a threshold by picking a shift and hoping. The experiment uses
an adaptive rule that walks the shift up and down until it settles where you are
right 79.4% of the time.

The rule, straight from the methods:

* start at 20 ms
* three right in a row and it divides by `c`
* one wrong and it multiplies by `c`
* `c` starts at 4, drops to 2 at the first turnaround, and to `sqrt(2)` two
  turnarounds after that
* stop at the sixth turnaround taken with the `sqrt(2)` step
* the threshold is the geometric mean of those last six turning points

One thing worth flagging, because it is the sort of thing that quietly eats an
experiment. The paper says "three-down one-up, which tracked the 79.4% point"
and then says the level drops after "two consecutive correct responses". Those
cannot both be true. Two-down converges on 70.7%, three-down on
`0.5 ** (1/3)` = 79.37%. The stated convergence point is the half that is not
ambiguous, so this code uses three-down.

Below, a fake listener with a threshold of exactly 3 ms is put through the real
rule.

In [ ]:
t = simulate(D, true_ms=3.0, seed=19)
lev = [x for x, _ in t.history]
ok = [c for _, c in t.history]
rep = t.report()

fig, ax = plt.subplots(figsize=(8.5, 3), constrained_layout=True)
ax.plot(lev, "-", color="0.6", lw=1)
ax.plot([i for i, c in enumerate(ok) if c], [x for x, c in zip(lev, ok) if c],
        "o", ms=4, color="#2ca02c", label="right")
ax.plot([i for i, c in enumerate(ok) if not c],
        [x for x, c in zip(lev, ok) if not c], "x", ms=6, color="#d62728",
        label="wrong")
ax.axhline(3.0, color="k", ls="--", lw=1, label="true threshold, 3.0 ms")
ax.axhline(rep["threshold_ms"], color="#1f77b4", lw=1.4,
           label=f"recovered, {rep['threshold_ms']:.2f} ms")
# mark the turning points, found the same way the rule finds them
dirs, turns, prev, nright = [], [], 0, 0
for i, (x, c) in enumerate(t.history):
    nright = nright + 1 if c else 0
    if c and nright < 3:
        continue
    here = -1 if c else 1
    nright = 0
    if prev and here != prev:
        turns.append(i)
    prev = here
for i in turns[-6:]:
    ax.plot(i, lev[i], "s", ms=9, mfc="none", color="#1f77b4")
ax.set_yscale("log"); ax.set_xlabel("trial"); ax.set_ylabel("shift, dT (ms)")
ax.legend(frameon=False, fontsize=8, ncol=2)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.show()
print(f"  {rep['n_trials']} trials, {100*rep['pc']:.0f}% correct, "
      f"{rep['n_reversals']} turnarounds")
print(f"  last six: {', '.join('%.2f' % r for r in rep['reversals'])}")

One run is noisy. That is why the paper takes at least four per condition per
listener and then a geometric mean. Here is what four runs buys you, and what
four hundred would.

In [ ]:
print(f"{'runs':>6}{'estimate':>11}{'spread (factor)':>18}")
for n in (1, 4, 16, 64):
    ests = []
    for block in range(400):
        th = [simulate(D, 3.0, seed=10000 + block * n + i).report()["threshold_ms"]
              for i in range(n)]
        ests.append(geomean(th))
    lo, hi = np.percentile(ests, [2.5, 97.5])
    print(f"{n:>6}{geomean(ests):>10.2f} {hi / lo:>17.2f}")
print("\n  the middle 95% of estimates spans that factor.")
print("  four runs is a factor of two. that is one listener's cell.")

---
## 7. The bit nobody has measured

Everything so far is the published experiment: together, or taking turns. But
those are two ends of a line, and the interesting part is the middle.

Take two streams that are perfectly synchronous and slide the high one a little
later. Five milliseconds. Ten. Twenty. At some point the single thing you were
hearing splits into two. Where?

Figure 8 of the paper runs exactly this sweep **in the model** and predicts a
smooth transition. It was never run on a listener. That is what `--mode sweep`
is for.

Both streams keep the same tempo here, 150 ms per beep, and only the phase
moves. That matters: the published asynchronous conditions change the tempo of
the A stream as well, and a tempo difference is a segregation cue in its own
right. Sliding the phase changes one thing only.

The tones are 75 ms in this mode rather than 100. Against a 150 ms period, 75 is
the one length where the far end of the sweep is exactly alternation: A fills
the first half of the period, B the second, no overlap and no gap. With 100 ms
tones the two would still overlap by 25 ms at the far end and the axis would
never reach its own endpoint. Figure 8 used 75 ms tones for the same reason.

In [ ]:
W = D.replace(mode="sweep")
W.validate()
rows = [seq(df_st=15.0, gap_a_ms=W.sweep_gap_ms, lag_ms=W.lag_ms(p), d=W)
        for p in W.sweep_pct]
piano(rows, [f"{p:.0f} %" for p in W.sweep_pct], d=W, mark_target=False,
      width=9.5)
print(f"  0 % is synchronous, 100 % is a lag of "
      f"{W.alternation_ms():.0f} ms, which is exact alternation")

Now listen down the ladder. Somewhere in here it stops being one beeping thing
and turns into two. For most people that happens sooner than they expect, and
the paper's own remark is that a 40 ms onset asynchrony, which is 53% of the way
here, already sounds like the alternating version.

There is no right answer to hear. That is the point: the number is what the
threshold measurement is for.

In [ ]:
ys = [seq(df_st=15.0, gap_a_ms=W.sweep_gap_ms, lag_ms=W.lag_ms(p), d=W)["y"]
      for p in W.sweep_pct]
display(players([(f"{p:>3.0f} %   lag {W.lag_ms(p):>4.1f} ms", y)
                 for p, y in zip(W.sweep_pct, ys)]))

The same ladder at 6 semitones, where the two tones are much closer together.
The prediction is that closer tones hang together for longer, so the split
should happen further along.

In [ ]:
ys6 = [seq(df_st=6.0, gap_a_ms=W.sweep_gap_ms, lag_ms=W.lag_ms(p), d=W)["y"]
       for p in W.sweep_pct]
display(players([(f"{p:>3.0f} %   lag {W.lag_ms(p):>4.1f} ms", y)
                 for p, y in zip(W.sweep_pct, ys6)]))

### What the model predicts

The paper's model takes the two channels, filters them at cortical rates, and
correlates them. Two channels whose activity rises and falls together belong to
one stream; two whose activity is out of phase belong to two. The number it
reports is the ratio of the second singular value of the correlation matrix to
the first: near zero for one stream, near one for two.

The full model is not in this notebook, but the quantity it is built on is easy
to compute and it is instructive. Below is the plain correlation between the
two channels' envelopes as the lag slides, which is the same idea without the
multi-scale filtering. Watch it go from +1 to about -1.

In [ ]:
from scipy.signal import hilbert

def channel_corr(lag_ms, df_st=15.0, d=None):
    d = d or W
    iv = seq(df_st=df_st, gap_a_ms=d.sweep_gap_ms, lag_ms=lag_ms, d=d)
    n = int(round(d.tone * d.fs / 1000))
    N = iv["y"].size
    ea, eb = np.zeros(N), np.zeros(N)
    env = np.abs(hilbert(S._tone(d, d.f_a, n)))
    for x in iv["a"]:
        i = int(round(x * d.fs / 1000)); ea[i:i + n] += env
    for x in iv["b"]:
        i = int(round(x * d.fs / 1000)); eb[i:i + n] += env
    lo, hi = int(0.25 * d.fs), int(0.95 * d.fs)
    a, b = ea[lo:hi], eb[lo:hi]
    a, b = a - a.mean(), b - b.mean()
    return float((a @ b) / np.sqrt((a @ a) * (b @ b)))

lags = np.linspace(0, W.alternation_ms(), 40)
fig, ax = plt.subplots(figsize=(6.4, 3.1), constrained_layout=True)
for df, col in ((6.0, "#1f77b4"), (15.0, "#d62728")):
    ax.plot(lags, [channel_corr(x, df) for x in lags], color=col, lw=1.6,
            label=f"{df:.0f} st")
ax.axhline(0, color="0.8", lw=0.8)
for p in W.sweep_pct:
    ax.axvline(W.lag_ms(p), color="0.92", lw=0.8, zorder=0)
ax.set_xlabel(f"lag of the B stream (ms).  0 = together, "
              f"{W.alternation_ms():.0f} = taking turns")
ax.set_ylabel("correlation between the two channels")
ax.set_title("what the model is looking at", loc="left", fontsize=10)
ax.legend(frameon=False, fontsize=9)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.show()
print("  the two separations give the same curve, because this measure knows")
print("  nothing about frequency. the listener's split point almost certainly")
print("  does depend on it, and that difference is the interesting result.")

---
## 8. Knobs

Every parameter of the experiment lives in one object, so you can turn any of
them and hear the result. Move the sliders and press Run Interact.

| knob | what it does |
|---|---|
| separation | how far B sits above A, in semitones |
| A gap | 50 makes the two streams synchronous, 30 and 70 make them drift |
| lag | slides the whole B stream, if you set the mode to sweep |
| shift | how far the last B beep moves. This is the thing being measured |
| tone | how long each beep is |
| B only | drop the A stream entirely, which is the control |

Some combinations do not fit and the code will say so rather than making
something silently wrong.

In [ ]:
import ipywidgets as Wd

@Wd.interact_manual(
    mode=Wd.Dropdown(options=["replicate", "sweep"], value="replicate",
                     description="mode"),
    df_st=Wd.SelectionSlider(options=[3, 6, 9, 12, 15, 18, 24], value=15,
                             description="separation"),
    gap_a_ms=Wd.SelectionSlider(options=[20, 30, 40, 50, 60, 70, 90], value=50,
                                description="A gap (ms)"),
    lag_pct=Wd.SelectionSlider(options=[0, 10, 20, 30, 40, 50, 65, 80, 100],
                               value=0, description="lag (%)"),
    dt_ms=Wd.SelectionSlider(options=[0, 1, 3, 6, 12, 20, 30, 40], value=20,
                             description="shift (ms)"),
    tone_ms=Wd.SelectionSlider(options=[50, 75, 100, 125], value=100,
                               description="tone (ms)"),
    b_only=Wd.Checkbox(value=False, description="B only"))
def playground(mode, df_st, gap_a_ms, lag_pct, dt_ms, tone_ms, b_only):
    kw = dict(mode=mode, ear="both")
    if mode == "sweep":
        kw.update(sweep_tone_ms=float(tone_ms), sweep_gap_ms=float(tone_ms))
    else:
        kw.update(tone_ms=float(tone_ms),
                  gap_b_ms=float(gap_a_ms) if gap_a_ms == 50 else 50.0)
    try:
        d = Design(**kw)
        d.validate()
        lag = d.lag_ms(lag_pct) if mode == "sweep" else 0.0
        gap = d.sweep_gap_ms if mode == "sweep" else float(gap_a_ms)
        std = seq(df_st=float(df_st), gap_a_ms=gap, lag_ms=lag, dt_ms=0.0,
                  b_only=b_only, d=d)
        sig = seq(df_st=float(df_st), gap_a_ms=gap, lag_ms=lag,
                  dt_ms=float(dt_ms), sign=1, b_only=b_only, d=d)
    except ValueError as e:
        return print(f"  that combination does not fit: {e}")
    display(HTML(f"<b>A {d.f_a:.0f} Hz, B {d.f_b(df_st):.0f} Hz, "
                 f"{d.tone:.0f} ms tones, lag {lag:.0f} ms, "
                 f"shift {dt_ms:g} ms</b>"))
    display(players([("nothing moved", std["y"]),
                     (f"last B beep {dt_ms:g} ms late", sig["y"])], fs=d.fs))
    piano([std, sig], ["standard", "signal"], d=d)

---
## 9. What is held constant

The listener is asked which sequence ended out of step. Anything else that
separates the two sequences is a way of being right without hearing the thing
the experiment is about, so all of it gets measured on the actual waveforms
rather than argued about.

Rows read standard / signal. This is the table that goes in the supplement.

In [ ]:
from audios.streaming.verify import verify, spectral, table
rows = [verify(D, df_st=15.0, gap_a_ms=g, lag_ms=0.0, b_only=False,
               dt_ms=3.0, n=12) for g in D.gap_a_ms]
rows.append(verify(D, df_st=15.0, gap_a_ms=50.0, lag_ms=0.0, b_only=True,
                   dt_ms=3.0, n=12))
print(table(rows, [spectral(D, x) for x in D.df_st]))

Reading it:

* **energy 0.0000 dB.** Moving a tone does not change how much sound there is.
* **every audible third-octave band 0.000 dB.** Bands more than 40 dB down hold
  the numerical skirt of the gate and are excluded, with the count shown.
* **first envelope difference** is the important row. The two sequences are
  identical, sample for sample, until the target. Nothing in the precursors
  gives it away.
* **shift forward on 50% of trials**, and the signal in interval 1 on 50%. Both
  balanced in blocks of four, not left to a coin. A run is about 45 trials, and
  an honest coin lands 60/40 or worse about a third of the time.
* **rendered dT matches asked dT to 0.0000 ms.** Onsets land on samples, and at
  48 kHz a sample is 0.021 ms against a floor of 0.25 ms.
* **splatter -73 dB.** The gate does not click. A click at an onset is the one
  artefact this task could not survive.

The last block is about the two tones themselves, and it contains the one
problem that cannot be engineered away.

### The confound that lives inside your ear

Two loud tones make your cochlea generate a third one that is not in the air,
at `2*fA - fB`. It exists only while the two overlap, so moving the target
changes how long it lasts, which means it tracks the signal.

| separation | B | `2*fA - fB` | |
|---|---|---|---|
| 6 st | 1414 Hz | **586 Hz** | audible |
| 9 st | 1682 Hz | **318 Hz** | audible |
| 15 st | 2378 Hz | -378 Hz | none generated |

No amount of care with the waveform removes it, because it is not in the
waveform. Two things make it liveable. The frequency ratio is 1.41 or wider,
well past the ~1.2 where these are strongest, so it is weak. And **15 semitones
generates none at all**, which turns the confound into a control: the paper's
result is that synchronous thresholds are small at *every* separation including
15 st, so if the effect held at 6 and 9 but vanished at 15, distortion would be
the reason. It does not.

Keep the level at 70 dB SPL and no higher, and say so in the methods.

In [ ]:
for df in D.df_st:
    s = spectral(D, df)
    cdt = s["cdt_hz"]
    note = ("none generated" if not s["cdt_in_band"]
            else f"{cdt:.0f} Hz, {s['cdt_erb_below_a']:.1f} critical bands below A")
    print(f"  {df:>2.0f} st   A 1000 Hz   B {s['f_b']:>7.1f} Hz   "
          f"ratio {s['f_b'] / D.f_a:.2f}   distortion product: {note}")

---
## 10. A whole experiment, from a listener who does not exist

Before anyone sits down for three hours you want to know that the analysis will
say the right thing when the data are right. The only way to check that is to
feed it data whose answer you already know.

Below, a fake listener whose thresholds are exactly the paper's values is put
through the real adaptive rule, and the output goes into the real analysis.
Nothing here is data.

In [ ]:
import csv, math, tempfile
from pathlib import Path
from audios.streaming.session import RUNS, cell_name, paths, run_list, write_meta
from audios.streaming.analyse import (PAPER, by_cell, key_tests, load, table,
                                      boundary)
from audios.streaming import plot as P

ROOT = Path(tempfile.mkdtemp())

def fake_subject(d, sid, truth, seed0=1000):
    rows = run_list(d, sid, 1)
    p = paths(ROOT, sid, 1)
    p["dir"].mkdir(parents=True, exist_ok=True)
    write_meta(p["meta"], d, dict(participant_id=sid, note="SIMULATED"), 1,
               blocks=dict(main=len(rows)))
    with p["runs"].open("w", newline="") as f:
        w = csv.DictWriter(f, RUNS, delimiter="\t", extrasaction="ignore")
        w.writeheader()
        for i, r in enumerate(rows):
            rep = simulate(d, truth(r), seed=seed0 + i).report()
            w.writerow({**{k: r.get(k, "") for k in
                           ("block", "run", "repeat", "kind", "df_st",
                            "gap_a_ms", "lag_ms", "pct", "b_only", "seed")},
                        "session": 1, "task": "streaming", "cell": cell_name(r),
                        "threshold_ms": round(rep["threshold_ms"], 4)
                        if rep["threshold_ms"] else "",
                        "n_trials": rep["n_trials"], "pc": round(rep["pc"], 4),
                        "n_reversals": rep["n_reversals"],
                        "clamped": rep["clamped"],
                        "at_floor": int(rep["at_floor"]),
                        "at_ceiling": int(rep["at_ceiling"]),
                        "why": rep["why"], "reversals": ""})
    return len(rows)

def truth_replicate(r):
    return PAPER.get((r["df_st"], None if r["b_only"] else r["gap_a_ms"]), 14.0)

n = fake_subject(D, "FAKE", truth_replicate)
print(f"  ran {n} adaptive tracks, {D.est_minutes:.0f} minutes of a real "
      f"listener's afternoon, in about a second")

In [ ]:
runs = load(ROOT, "FAKE")
cells = by_cell(runs)
print(table(D, cells, key_tests(D, runs)))

In [ ]:
P.figure2(D, cells, ROOT / "fig2.png")
from IPython.display import Image
display(Image(str(ROOT / "fig2.png")))

The grey band is the published data. Filled squares, the synchronous
condition, sit on it. Everything else sits four to six times higher, and the
B-only crosses land on top of the asynchronous conditions rather than above
them, which is the claim.

Look at the error bars, though. **Four runs in one listener gives a 95%
interval spanning roughly a factor of two.** One listener cannot reproduce that
figure point by point. What comes out solidly is the pattern, and the pattern is
what the paper is about. Plan for the pattern; pool listeners for the points.

### And the sweep

Same machinery, with a fake listener who comes apart at a lag of 25 ms.

In [ ]:
def truth_sweep(r):
    lo = PAPER[(r["df_st"], 50.0)]
    hi = PAPER[(r["df_st"], None)]
    z = (r["lag_ms"] - 25.0) / 6.0
    return float(lo * (hi / lo) ** (1 / (1 + math.exp(-z))))

fake_subject(W, "FAKESWEEP", truth_sweep, seed0=7000)
runs_w = load(ROOT, "FAKESWEEP")
cells_w = by_cell(runs_w)
print(table(W, cells_w, key_tests(W, runs_w)))
for df in sorted({c["df_st"] for c in cells_w}):
    b = boundary(cells_w, df)
    print(f"\n  {df:>2.0f} st: crosses twice the synchronous threshold at a lag "
          f"of {b['lag_ms']:.0f} ms" if np.isfinite(b["lag_ms"])
          else f"\n  {df:>2.0f} st: {b['why']}")

In [ ]:
P.sweep(W, cells_w, ROOT / "sweep.png")
display(Image(str(ROOT / "sweep.png")))

The true split point was 25 ms and the recovery is within about ten. If the
split point is the number you want to publish, that is not good enough, and the
fix is more runs per cell in the sweep than the replication needs, or more
listeners, or both. Better to find that out here than after twenty people have
each given you an afternoon.

---
## 11. Running it on a person

None of the above needs a lab. The experiment does: a quiet room, calibrated
headphones, and a terminal, because the runner reads keypresses and plays sound
with millisecond timing, which a notebook cannot do.

```bash
git clone https://github.com/MeysamAmirsardari/Rate_RNN.git
cd Rate_RNN

python -m audios.streaming selftest        # before anyone sits down
python -m audios.streaming check           # the table from section 9
python -m audios.streaming calibrate       # 70 dB SPL, once

python -m audios.streaming run S01                 # the replication, 48 tracks
python -m audios.streaming run S01 --mode sweep    # the extension, 72 tracks
python -m audios.streaming run S01 --resume        # pick up an interrupted one

python -m audios.streaming analyse S01
python -m audios.streaming analyse S01 --mode sweep
```

The replication is about three hours and the sweep is longer. That is not one
sitting and it is not meant to be: the paper's listeners were experienced and
did at least four runs in each of twelve conditions. Run it across sessions and
the analysis pools them. Runs are ordered so that every condition gets its first
repeat before any gets its second, which spreads practice and fatigue evenly
instead of dumping both on whichever condition happened to be last.

Everything lands in a BIDS-style tree: one row per trial with the shift that was
presented, one row per track with its turning points, and a JSON beside them
holding the full design, the participant record and the git commit, so any
stimulus in the session can be rebuilt from its seed.

The rationale, every parameter, and a list of what the design cannot settle are
in
[`audios/streaming/README.md`](https://github.com/MeysamAmirsardari/Rate_RNN/blob/main/audios/streaming/README.md).

---
## What to listen for, if you only have five minutes

Go back to section 2 and play those two sounds. Two tones more than an octave
apart, taking turns, sound like two things. The same two tones, at the same
time, sound like one. That single contrast is the paper.

Then go to section 7 and walk down the ladder. Somewhere between together and
taking turns, the one thing becomes two. Nobody has measured where.